# Сегментація дороги — навчання в Colab

**Перед стартом:** меню *Середовище виконання → Змінити тип середовища → GPU (T4)*.

На Google Drive має бути папка проєкту (за замовчуванням `MyDrive/road_seg`) з файлами `extract_frames.py`, `train_seg.py`, `seg_video.py`. Відео можуть лежати в будь-якій папці Drive.

Запускай клітинки по черзі зверху вниз (кнопка ▶ зліва від клітинки).

In [ ]:
!nvidia-smi -L || echo 'GPU немає: Середовище виконання -> Змінити тип -> GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Налаштування
Вкажи, де лежить проєкт і відео, та які відео на навчання, а які на фінальну перевірку.

In [ ]:
import os, glob

PROJECT = '/content/drive/MyDrive/road_seg'   # папка з .py файлами
VIDEOS  = '/content/drive/MyDrive/Track 1'    # ярлик на папку з відео (Shared with me -> Add shortcut)

# назви без .mp4: 9 на навчання, 3 на фінальну перевірку (різні типи сцен)
TRAIN_VIDEOS = ['dev_0001', 'dev_0002', 'dev_0003', 'dev_0004', 'dev_0006',
                'dev_0007', 'dev_0009', 'dev_0010', 'dev_0011']
TEST_VIDEOS  = ['dev_0005', 'dev_0008', 'dev_0012']

os.chdir(PROJECT)
print('Файли проєкту:', sorted(os.listdir(PROJECT)))
print('Відео на Drive:', sorted(os.path.basename(v) for v in glob.glob(VIDEOS + '/*')))

In [ ]:
!pip -q install segmentation-models-pytorch

## Крок 0. Готова розмітка від організаторів
Архів `seg-*.zip` з папки з відео: 301 кадр з масками. Беремо з них лише клас «дорога».

In [ ]:
z = sorted(glob.glob(f'{VIDEOS}/seg*.zip'))
print('Архів:', z)
if z:
    !rm -rf /content/seg && mkdir -p /content/seg && unzip -oq "{z[0]}" -d /content/seg
    !python convert_given.py /content/seg --data data --name given
else:
    print('Архів seg*.zip не знайдено у папці з відео')

## Крок 1. Нарізати кадри з відео для навчання
Кадр кожні 10 с (~30 з кожного відео). Ляжуть у `road_seg/data/images` на Drive і запакуються в `images.zip` — його скачай на ноутбук для розмітки.

In [ ]:
!python extract_frames.py "$VIDEOS" --only {' '.join(TRAIN_VIDEOS)} --out data/images --every 10
!cd data && rm -f ../images.zip && zip -qr ../images.zip images -x 'images/given__*' && echo 'Готово: road_seg/images.zip на Drive'

## Крок 2. Розмітка (на ноутбуці)
1. Скачай `road_seg/images.zip` з Drive, розпакуй у `~/lanes/road_seg/data/`.
2. Запусти `python label_tool.py` і розміть кадри: 15–25 з кожного відео достатньо (готові 301 кадр уже є).
3. Запакуй папку `data/masks` у `masks.zip` і закинь у `road_seg/` на Drive.

Потім запусти клітинку нижче.

In [ ]:
!cd data && unzip -oq ../masks.zip && echo "Масок: $(ls masks/*.png | wc -l)"

## Крок 3. Навчання
Найкраща модель зберігається після кожної епохи в `road_seg/models/road_seg.pt`, тож якщо сесія обірветься — нічого не пропаде.

In [ ]:
!python train_seg.py --data data --out models/road_seg.pt --epochs 20 --batch 16 --val-videos dev_0010

## Крок 4. Перевірка на відео, яких модель не бачила
Результати — у `road_seg/results/` на Drive.

In [ ]:
for v in TEST_VIDEOS:
    path = next(iter(glob.glob(f'{VIDEOS}/{v}.*')), None)
    if path:
        !python seg_video.py "$path" --model models/road_seg.pt --out results
    else:
        print('Не знайдено', v)

## Крок 5. Забрати модель на ноутбук
Скачай `road_seg/models/road_seg.pt` з Drive у `~/lanes/road_seg/models/` і запускай локально:

```
python seg_video.py моє_відео.mp4 --model models/road_seg.pt --show
```